# OPC UA to Microsoft Fabric Real-Time Intelligence - step by step

This notebook walks through streaming machine data from **OPC UA** servers into a **Fabric Eventstream**, using `src/opcua_to_eventstream.py`.

```
Machines (OPC UA servers) --OPC UA--> this PC (bridge) --outbound 443--> Fabric Eventstream --> Eventhouse (bronze > silver > gold)
```

**Where to run it:** in VS Code or Jupyter **on the bridge PC/VM inside the machine network** (for example the shop-floor VLAN). It must be able to reach the machines, so it will not work from a Fabric notebook.

| Step | What you do |
|---|---|
| 0 | Install the Python packages |
| 1 | Load the settings |
| 2 | Browse the OPC UA server to find NodeIds |
| 3 | Read the configured tags once |
| 4 | Send one test event to Fabric |
| 5 | Stream for 15 seconds without sending (dry run) |
| 6 | Stream to Fabric for 60 seconds |
| 7 | (Optional) Keep streaming in the background |
| 8 | Check the data in Fabric |

**Before you start (one-off, in Fabric):**
1. Create an Eventhouse and run `kql/medallion.kql`.
2. Create an Eventstream with a **custom endpoint** source, then **Publish**.
3. Copy the connection string from **Details > Event Hub > SAS Key Authentication** into `.env`.
4. Add an **Eventhouse** destination with **Event processing before ingestion** to the table `opcua_bronze` (JSON format), then **Publish**.

> Sample code, provided as-is for learning and prototyping. Don't store secrets in this notebook - keep them in `.env`.

## Step 0 - Install the packages

In [ ]:
%pip install -q -r ../requirements.txt

## Step 1 - Load the settings

Settings come from `.env` in the repository root (copy `.env.example`). With no machine settings, the bridge uses the **OPC PLC simulator** at `opc.tcp://localhost:50000` (start it with `docker compose up -d`).

In [ ]:
import sys, json, logging, asyncio
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import opcua_to_eventstream as bridge

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
bridge.hide_known_sdk_noise()

cfg = bridge.load_config(str(REPO_ROOT / ".env"))

# Optional overrides for this session (keep secrets in .env, not here)
# cfg.machines[0].endpoint = "opc.tcp://<machine-host>:4840"
# cfg.run_id = "DOE-2026-10-06-R01"

for m in cfg.machines:
    print(f"{m.machine_id}: {m.endpoint} | {len(m.nodes)} tags | security: {'on' if m.security else 'off'}")
if cfg.connection_string:
    target = "SAS connection string"
elif cfg.namespace:
    target = f"Entra ID ({cfg.namespace})"
else:
    target = "NOT SET - steps 4, 6 and 7 need it"
print("Fabric target:", target, "| run ID:", cfg.run_id)

## Step 2 - Browse the OPC UA server

Lists variables with their **NodeId**, browse path and current value, so you can choose the tags to stream. Nodes you aren't allowed to read show as `<unreadable>`. Set `BROWSE_ROOT` to start lower in the tree (for the simulator, `"ns=3;s=OpcPlc"`). Copy the NodeIds you need into `NODE_MAP_JSON` (one machine) or `config/machines.json` (several machines).

In [ ]:
BROWSE_ROOT = None   # e.g. "ns=3;s=OpcPlc" for the OPC PLC simulator's tags
rows = await bridge.browse_variables(cfg.machines[0], max_depth=4, max_nodes=200, root=BROWSE_ROOT)
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(f"{r['nodeId']:<45} {r['path']:<60} {r['value']!r}")

## Step 3 - Read the configured tags once

A quick check that the NodeIds and permissions are correct. Any problem appears in the `error` column.

In [ ]:
values = await bridge.read_current_values(cfg.machines[0])
for v in values:
    print(json.dumps(v, default=str))

## Step 4 - Send one test event to Fabric

Checks the Fabric connection without OPC UA. The event should appear in the Eventstream live view within a few seconds, with `tag = "ConnectivityTest"`, and in `opcua_bronze` within about a minute.

In [ ]:
event = await bridge.send_test_event(cfg)
print("Sent:", json.dumps(event, indent=2))

## Step 5 - Stream for 15 seconds without sending (dry run)

Prints the events that would be sent, so you can check the shape and values first.

In [ ]:
stats = await bridge.run(cfg, duration=15, dry_run=True)
stats

## Step 6 - Stream to Fabric for 60 seconds

In [ ]:
stats = await bridge.run(cfg, duration=60)
stats   # 'sent' should match 'received'; 'spooled' > 0 means Fabric could not be reached

## Step 7 - (Optional) Keep streaming in the background

Starts the bridge as a background task so you can keep using the notebook. Run the next cell to stop it. For anything longer than a test, run the script as a service instead (see the README).

In [ ]:
stop = asyncio.Event()
bridge_task = asyncio.create_task(bridge.run(cfg, stop=stop))
print("Bridge running in the background - run the next cell to stop it.")

In [ ]:
stop.set()
print(await bridge_task)

## Step 8 - Check the data in Fabric

Run these in a KQL queryset on the Eventhouse (more in `kql/verify.kql`):

```kusto
// Events in the last 15 minutes, per machine and tag
opcua_bronze
| where ingestion_time() > ago(15m)
| summarize events = count(), lastEvent = max(sourceTimestamp) by machineId, tag

// End-to-end delay from machine to Eventhouse (seconds)
opcua_bronze
| where ingestion_time() > ago(15m)
| extend delaySec = datetime_diff('second', ingestion_time(), sourceTimestamp)
| summarize avgDelaySec = avg(delaySec), p95DelaySec = percentile(delaySec, 95) by machineId

// Statistics per experiment run (gold layer)
opcua_gold_1min
| where isnotempty(runId)
| summarize avgValue = avg(avgValue), samples = sum(samples) by runId, machineId, tag
```

## Next steps

* **Medallion layers:** `kql/medallion.kql` creates the silver table (update policy), a de-duplicated view, 1-minute gold statistics and OneLake availability for the silver table. Materialised views stay in the Eventhouse (query them with KQL); see the README.
* **Production:**
  * use Microsoft Entra ID instead of the SAS key;
  * use OPC UA `SignAndEncrypt` with trusted certificates;
  * run the script as a service;
  * monitor the logs and the Eventstream.
* **Linking runs:** set `RUN_ID` for each experiment so telemetry can be joined with DOE sheets and specimen data.